**Install SDK**

With this SDK you can perform Business Data Cloud Connect operations such as:

- Create or update shares
- Create or update shares CSN
- Publish or unpublish Data Products

In [0]:
# Code 1
%pip install sap-bdc-connect-sdk
#%restart_python

**Create a client**

-   DatabricksClient receives dbutils as a parameter, which is a Databricks utility that can be used inside the Databricks notebooks
-   BdcConnectClient receives the DatabricksClient as a parameter to get information from the Databricks environment (e.g. secrets, api_token, workspace_url_base)

In [0]:
# Code 2
from bdc_connect_sdk.auth import BdcConnectClient
from bdc_connect_sdk.auth import DatabricksClient

databricks_client_name = "sap-business-data-cloud"
databricks_client = DatabricksClient(dbutils, databricks_client_name)
bdc_connect_client = BdcConnectClient(databricks_client)

**Create or update share**

A share is a mechanism for distributing and accessing data across different systems. Creating or updating a share involves including specific attributes, such as @openResourceDiscoveryV1, in the request body, aligning with the Open Resource Discovery protocol. This procedure ensures that the share is properly structured and described according to specified standards, facilitating effective data sharing and management.

In [0]:
# Code 3

#programmatially import the user name, strip out the sapexperienceacademy.com part, and ensure it is lower case
username = spark.sql("SELECT current_user() AS username").collect()[0]['username']
username_substr = username.split('@')[0]
username_lc = username_substr.lower()

#create a share name to match the Delta Share previoulsy created by this user
share_name = f"company_code_clustering_share_jblandon"

#validate that the share exists
delta_shares_df = spark.sql("SHOW SHARES")
if share_name in [row['share'] for row in delta_shares_df.select('share').collect()]:
    existing_recipients = [
        row.recipient
        for row in spark.sql(f"SHOW GRANTS ON SHARE {share_name}").collect()
    ]
    
    #validate that the databricks client is a recipient
    if databricks_client_name in existing_recipients:
        #set the data product title
        the_title = f"Company Code Clustering Data Product From {username_lc}"
        
        open_resource_discovery_information = {
            "@openResourceDiscoveryV1": {
               "title": the_title,
               "shortDescription": "Data asset for company code clustering from SAP Databricks",
               "description": "This data product contains the data used for the company clustering prediction model. A hyperparameter optimization is performed on the cleased data with the help of the Bayesian Search optimization. For the clustering we use the Affinity Propagation algorithm.After the hyperparameter optimization is performed, the model with the highest silhouette score is applied as the best model to the prepared clustering dataset. The resulting clustering labels are stored and merged together with a T-SNE based representation of the input dataset."
            }
        }
        
        response = bdc_connect_client.create_or_update_share(
           share_name,
           open_resource_discovery_information
         )
        print(f"[REQUEST] Create or update share request was executed and returned {response}")
    else:
        raise RuntimeError(f"ERROR: Delta Share '{share_name}' is not shared with '{databricks_client_name}'.")
else:
     raise RuntimeError(f"ERROR: Delta Share '{share_name}' does not exist.")

**Create or update share CSN**

The CSN serves as a standardized format for configuring and describing shares within a network. To create or update the CSN for a share, it's advised to prepare the CSN content in a separate file and include this content in the request body. This approach ensures accuracy and compliance with the CSN interoperability specifications, facilitating consistent and effective share configuration across systems.

In [0]:
# Code 4
from bdc_connect_sdk.utils import csn_generator
from bdc_connect_sdk.auth import BdcConnectClient

bdc_connect_client = BdcConnectClient(databricks_client)
csn_schema = csn_generator.generate_csn_template(share_name)

response = bdc_connect_client.create_or_update_share_csn(
    share_name,
    csn_schema
)
print(f"[REQUEST] Create or update CSN request was executed and returned {response if response else 'OK'}")


**Publish a Data Product**

A Data Product is an abstraction that represents a type of data or data set within a system, facilitating easier management and sharing across different platforms. It bundles resources or API endpoints to enable efficient data access and utilization by integrated systems. Publishing a Data Product allows these systems to access and consume the data, ensuring seamless communication and resource sharing.

In [0]:
# Code 5
from bdc_connect_sdk.auth import BdcConnectClient

bdc_connect_client = BdcConnectClient(databricks_client)
response = bdc_connect_client.publish_data_product(
    share_name
)
print(f"[REQUEST] Publish Data Product request was executed and returned {response if response else 'OK'}")